# 06 - Role-dropout fusion LoRA (the fine-tuned generator)

Team One Sheep Army. Fine-tunes the SDXL UNet with a LoRA so that the generator **obeys the governance gates**.

Each sample is an aligned quadruple from `pilot_v1`: target photo, sketch, reference (a different object with the target's colour and material), caption. Per step one regime is drawn:

| regime | caption | reference | teaches |
|---|---|---|---|
| full (40%) | as is | full | use all three |
| text_colour (25%) | states the colour | greyscale only | colour from the text, texture from the reference |
| ref_colour (25%) | colour words removed | full | colour from the reference |
| drop (10%) | negative prompt | none | keeps classifier-free guidance valid |

Frozen: VAE, text encoders, IP-Adapter, ControlNet, UNet base weights. Trained: LoRA on the UNet attention layers.

**Run the memory test first.** If it fails, lower `RES` to 384, set `RANK = 4`, `SCOPE = "up"`.

In [ ]:
# ---- CONFIG + SETUP (same in notebooks 05, 06, 07) ----
DRIVE_ROOT = "/content/drive/MyDrive/sketchy_dataset_one_sketch"
ZIP_PATH = f"{DRIVE_ROOT}/pilot_v1.zip"
LOCAL_DATA = "/content/pilot_v1"
GOV_DIR = f"{DRIVE_ROOT}/governance"            # governance.pt, cached features, fusion LoRA all live here
# The code lives in the repo's src/ folder (cloned below). To use a copy on Drive instead, set REPO_DIR to it.
REPO_URL = "https://github.com/Harshitha-282007/IE643_final.git"
REPO_BRANCH = "governed-pipeline"                # change to "master" once the branch is merged
REPO_DIR = "/content/repo"

import os, sys, glob, json, zipfile
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/src/pipeline.py"), "src/pipeline.py not found: push the new code or point REPO_DIR at it"
sys.path.insert(0, f"{REPO_DIR}/src")
os.makedirs(GOV_DIR, exist_ok=True)
if not glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_DATA)
DATA_DIR = os.path.dirname(glob.glob(f"{LOCAL_DATA}/**/metadata.jsonl", recursive=True)[0])
BASE_ROWS = [json.loads(l) for l in open(f"{DATA_DIR}/metadata.jsonl") if l.strip()]
print("data:", DATA_DIR, "| base rows:", len(BASE_ROWS))

In [ ]:
!pip uninstall -y torchao -q
!pip install -q -U diffusers transformers accelerate safetensors peft bitsandbytes
RES, RANK, SCOPE = 512, 8, "all"       # fallback: 384, 4, "up"
MAX_STEPS, ACCUM, LR = 800, 4, 1e-4
LORA_DIR = f"{GOV_DIR}/fusion_lora"
import torch
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# ---- 1. memory test in a separate process (3 optimizer steps, nothing saved) ----
!cd {REPO_DIR}/src && python train_fusion_lora.py --data "{DATA_DIR}" --out "{LORA_DIR}" --res {RES} --rank {RANK} --scope {SCOPE} --accum {ACCUM} --max_steps 3 --memory_test

In [ ]:
# ---- 2. train (saves every 200 steps; about 1.5-2 s per forward pass on a T4) ----
from train_fusion_lora import train
P, losses = train(DATA_DIR, LORA_DIR, res=RES, rank=RANK, scope=SCOPE, lr=LR, max_steps=MAX_STEPS, accum=ACCUM)
import numpy as np, matplotlib.pyplot as plt
k = 50
plt.figure(figsize=(7, 3)); plt.plot(np.convolve(losses, np.ones(k) / k, mode="valid")); plt.xlabel("forward pass"); plt.ylabel(f"loss (mean of {k})")
plt.title("fusion LoRA training loss"); plt.savefig(f"{LORA_DIR}/loss.png", dpi=110, bbox_inches="tight"); plt.show()
print("copy", f"{LORA_DIR}/fusion_lora.safetensors", "to models/ in the repo")

In [ ]:
# ---- 3. sample check on held-out photos: same seed, with and without the LoRA, in two regimes ----
from PIL import Image
from governance import labels as L
from governance.features import ClipL
P.clip = ClipL("cuda")
test = [r for r in BASE_ROWS if r["split"] == "test"][:4]
op = lambda r, k: Image.open(f"{DATA_DIR}/{r[k]}")
items = P.make_items([dict(key=r["id"], sketch=op(r, "sketch"), ref=op(r, "reference"), caption=r["caption"],
                           obj=r["category"], seed=0) for r in test])
cols = [("no LoRA | reference governs colour", False, dict(g_obj=0, g_col=0, g_mat=0)),
        ("LoRA | reference governs colour", True, dict(g_obj=0, g_col=0, g_mat=0)),
        ("no LoRA | text governs colour", False, dict(g_obj=0, g_col=1, g_mat=0)),
        ("LoRA | text governs colour", True, dict(g_obj=0, g_col=1, g_mat=0))]
fig, ax = plt.subplots(len(items), 3 + len(cols), figsize=(3 * (3 + len(cols)), 3 * len(items)))
for i, (it, r) in enumerate(zip(items, test)):
    for j, (im, t) in enumerate([(it["sketch"], "sketch"), (it["ref"], "reference"), (op(r, "target"), "target")]):
        ax[i, j].imshow(im); ax[i, j].set_title(t if i == 0 else "", fontsize=8)
    for j, (name, lora, g) in enumerate(cols):
        P.use_lora(lora)
        ax[i, 3 + j].imshow(P.run(it, gates=g)); ax[i, 3 + j].set_title(name if i == 0 else "", fontsize=8)
    ax[i, 0].set_ylabel(r["caption"][:40], fontsize=7)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f"{LORA_DIR}/sample_check.png", dpi=100); plt.show()
print("Look for: with 'text governs colour' the object keeps the caption's colour but takes the reference's surface.")